# Dialogue systems & chatbots — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def softmax(x, axis=-1):
    x=np.asarray(x, dtype=float); x=x-x.max(axis=axis, keepdims=True); e=np.exp(x); return e/e.sum(axis=axis, keepdims=True)
def sigmoid(x): return 1/(1+np.exp(-np.asarray(x, dtype=float)))
def normalize(v):
    v=np.asarray(v, dtype=float); n=np.linalg.norm(v); return v/(n if n else 1)
def edit_distance(a,b):
    D=np.zeros((len(a)+1,len(b)+1), dtype=int); D[:,0]=np.arange(len(a)+1); D[0,:]=np.arange(len(b)+1)
    for i in range(1,len(a)+1):
        for j in range(1,len(b)+1):
            D[i,j]=min(D[i-1,j]+1, D[i,j-1]+1, D[i-1,j-1]+(a[i-1]!=b[j-1]))
    return D
print('setup ok')

## Maintain conversational state while selecting safe, useful next actions
The examples show intent probabilities, slot filling, dialogue state, retrieval response selection and safety fallback.

In [ ]:
intents=['book','cancel','chitchat']; p=softmax([2.0,.5,.1])
plt.figure(figsize=(5,3)); plt.bar(intents,p); plt.title('Intent classifier routes the turn')
assert intents[int(np.argmax(p))]=='book'

In [ ]:
slots={'date':1,'city':0,'time':1}; plt.figure(figsize=(5,3)); plt.bar(list(slots),list(slots.values())); plt.title('Slot filling tracks required information')
assert sum(slots.values())==2

In [ ]:
turns=np.arange(4); filled=np.array([0,1,2,3]); plt.figure(figsize=(5,3)); plt.step(turns,filled,where='mid'); plt.title('Dialogue state accumulates across turns')
assert filled[-1]>filled[0]

In [ ]:
q=np.array([1.,0.]); resp=np.array([[.9,.1],[.2,.8],[.7,.3]]); sims=resp@q/(np.linalg.norm(resp,axis=1)*np.linalg.norm(q))
plt.figure(figsize=(5,3)); plt.bar(['r0','r1','r2'],sims); plt.title('Retrieval chatbot chooses nearest response')
assert int(np.argmax(sims))==0

In [ ]:
safe=np.array([.95,.6,.2]); fallback=safe<.5
plt.figure(figsize=(5,3)); plt.bar(['turn0','turn1','turn2'],safe); plt.axhline(.5,ls='--',c='r'); plt.title('Safety threshold triggers fallback')
assert fallback[-1] and not fallback[0]